# 01. Data

This notebook inspects the raw data before it is used: structure, missing values, label, leakage risk, and sample consistency. All statistics are computed directly from the data; none are typed by hand.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import re

import pandas as pd

from src.config import load_params
from src.data.load import data_hash, load_raw, to_csv_text
from src.data.sample import make_sample
from src.data.validate import banner, validate

params = load_params()
print("data_source:", params["data_source"])
print("synthetic banner:", banner(params))

data_source: real_public_ibm_telco
synthetic banner: None


## Problem (excerpt from the README)

The block below is read directly from `README.md`, not rewritten here.

In [3]:
readme = (ROOT / "README.md").read_text(encoding="utf-8")
block = re.search(r"## Problem\n(.*?)(?=\n## )", readme, re.S).group(1).strip()
print(block)

- **Domain and relationship.** A subscription telecom service. The relationship is mixed: month-to-month customers (effectively non-contract) and customers on one-year or two-year contracts.
- **Churn definition and window.** Churn is `Churn = Yes` in the file. The data has one row per customer and no dates, so the prediction window is undefined and the project uses a non-temporal variant (see `docs/adaptation_path_a.md`).
- **Population at risk.** All rows in the file, because churn status before the snapshot is not observed.
- **Decision that depends on the score.** Customers whose calibrated `p_churn` is above the break-even threshold `p*` and who rank within campaign capacity `k` are contacted. Capacity and population per period are in `assumptions.yaml`.
- **CLV, intervention cost, save_rate.** All come from `assumptions.yaml`. While `save_rate_source = assumed`, all business value is conditional.


## Load and validate

In [4]:
raw = load_raw("raw", params)
report = validate(raw, params)
print("rows x columns:", raw.shape)
print("data_hash:", data_hash(raw)[:16])
print("validation ok:", report.ok)
assert report.ok, report.errors

rows x columns: (7043, 21)
data_hash: deae3674baa798c6
validation ok: True


## Column structure

In [5]:
def kind(s):
    if pd.api.types.is_integer_dtype(s):
        return "integer"
    if pd.api.types.is_float_dtype(s):
        return "float"
    return "text"


info = raw.apply(kind).to_frame("type")
info["missing"] = raw.isna().sum()
print(info.to_string())

                     type  missing
customerID           text        0
gender               text        0
SeniorCitizen     integer        0
Partner              text        0
Dependents           text        0
tenure            integer        0
PhoneService         text        0
MultipleLines        text        0
InternetService      text        0
OnlineSecurity       text        0
OnlineBackup         text        0
DeviceProtection     text        0
TechSupport          text        0
StreamingTV          text        0
StreamingMovies      text        0
Contract             text        0
PaperlessBilling     text        0
PaymentMethod        text        0
MonthlyCharges      float        0
TotalCharges        float       11
Churn                text        0


## Missing TotalCharges

Validator rule: `TotalCharges` may be missing only when `tenure` = 0 (a new customer with no bills yet). The cell below verifies that the real data follows this rule.

In [6]:
nulls = raw[raw["TotalCharges"].isna()]
print("rows with missing TotalCharges:", len(nulls))
print("tenure values on those rows:", sorted(nulls["tenure"].unique().tolist()))
print("label on those rows:")
print(nulls["Churn"].value_counts().to_string())
assert len(nulls) > 0 and (nulls["tenure"] == 0).all()

rows with missing TotalCharges: 11
tenure values on those rows: [0]
label on those rows:
Churn
No    11


## Label

In [7]:
sc = params["schema"]
y = (raw[sc["label_column"]] == sc["positive_label"]).astype(int)
print(raw[sc["label_column"]].value_counts().to_string())
print("churn rate:", round(y.mean(), 4))
assert raw[sc["id_column"]].is_unique

Churn
No     5174
Yes    1869
churn rate: 0.2654


## Churn by segment

In [8]:
for col in ["Contract", "InternetService", "PaymentMethod"]:
    seg = raw.groupby(col)[sc["label_column"]].agg(
        n="size", churn_rate=lambda s: round((s == sc["positive_label"]).mean(), 3)
    )
    print(seg.to_string())
    print()

                   n  churn_rate
Contract                        
Month-to-month  3875       0.427
One year        1473       0.113
Two year        1695       0.028

                    n  churn_rate
InternetService                  
DSL              2421       0.190
Fiber optic      3096       0.419
No               1526       0.074

                              n  churn_rate
PaymentMethod                              
Bank transfer (automatic)  1544       0.167
Credit card (automatic)    1522       0.152
Electronic check           2365       0.453
Mailed check               1612       0.191



## Leakage risk: tenure and TotalCharges

Both columns accumulate until churn is observed. The difference in their distributions between labels below shows why both are flagged in `params.yaml`.

In [9]:
flagged = params["leakage"]["flagged_columns"]
print("flagged columns:", flagged)
print("policy:", params["leakage"]["policy"])
for col in flagged:
    print()
    print(raw.groupby(sc["label_column"])[col].describe().round(2).to_string())

flagged columns: ['tenure', 'TotalCharges']
policy: exclude

        count   mean    std  min   25%   50%   75%   max
Churn                                                   
No     5174.0  37.57  24.11  0.0  15.0  38.0  61.0  72.0
Yes    1869.0  17.98  19.53  1.0   2.0  10.0  29.0  72.0

        count     mean      std    min     25%      50%      75%      max
Churn                                                                    
No     5163.0  2555.34  2329.46  18.80  577.83  1683.60  4264.12  8672.45
Yes    1869.0  1531.80  1890.82  18.85  134.50   703.55  2331.30  8684.80


## Sample versus raw data

In [10]:
sample = load_raw("sample", params)
print("sample rows:", len(sample))
print("sample validation ok:", validate(sample, params).ok)
y_s = (sample[sc["label_column"]] == sc["positive_label"]).astype(int)
print("churn rate difference sample - raw:", round(y_s.mean() - y.mean(), 4))
committed = (ROOT / params["paths"]["sample"]).read_text(encoding="utf-8")
same = committed == to_csv_text(make_sample(raw, params))
print("sample.csv matches the generator:", same)
assert same

sample rows: 1000
sample validation ok: True
churn rate difference sample - raw: -0.0004
sample.csv matches the generator: True


## Findings for the next parts

- One row per customer, unique ids, no dates: the non-temporal variant applies.
- `TotalCharges` is missing only for new customers (`tenure` = 0); the validator enforces this rule.
- The label maps `Churn == Yes` to 1. The positive class is the minority, so the primary metric is PR-based, not accuracy.
- `tenure` and `TotalCharges` differ sharply between labels because they accumulate until churn; the default policy excludes them from the features.
- The stratified sample keeps the churn rate close to that of the raw data.